# Faster-Whisper ASR Prototype

Upload prerecorded WAV, MP3, M4A, or WebM product-query clips, transcribe them with `faster-whisper`, and evaluate product terms and word error rate.


## 1. Install dependencies


In [1]:
!pip install -q faster-whisper jiwer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 94.1 MB/s eta 0:00:00


## 2. Check runtime and load the model

For Colab, select **Runtime → Change runtime type → T4 GPU** when available. The code falls back to CPU automatically.


In [2]:
import torch
from faster_whisper import WhisperModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
MODEL_SIZE = "small.en"

print(f"Device: {DEVICE}")
print(f"Compute type: {COMPUTE_TYPE}")
print(f"Model: {MODEL_SIZE}")

model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)


Device: cuda
Compute type: float16
Model: small.en


## 3. Upload prerecorded audio files


Two audio sets were used to evaluate Whisper ASR:

- **Original prototype:** 10 previously recorded product queries (`query01.wav`–`query10.wav`) used to evaluate baseline transcription performance.
- **Voicemod evaluation:** 5 standardized product queries recorded under five voice conditions: the original voice and four Voicemod-modified voices (A–D). This produced 25 recordings.
- The four Voicemod conditions represent different voice characteristics:
  - **Voice A – Agatha:** older/elderly-sounding voice
  - **Voice B – Joe:** older/elderly-sounding voice
  - **Voice C – Yuto:** younger-sounding voice
  - **Voice D – Mei:** younger-sounding voice
- The Voicemod queries include challenging e-commerce terms such as brand names, product/model names, prices, and quantities.
- All voice conditions use the same five reference sentences so transcription accuracy can be compared consistently across voice conditions.
- Audio filenames follow the format `query##_condition.wav` (for example, `query02_A.wav`).

The Voicemod recordings are used to evaluate robustness across different voice conditions using **Word Error Rate (WER)** and **product-term accuracy**. Both `small.en` and `medium.en` Whisper models are tested on the same recordings to determine whether increasing model size improves recognition accuracy.

Because the Voicemod conditions are synthetic voice transformations rather than recordings from speakers of verified ages or accents, the labels describe how the voices **sound** rather than demographic characteristics of actual speakers.

In [3]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

SUPPORTED_EXTENSIONS = {".wav", ".mp3", ".m4a", ".webm", ".flac", ".ogg"}
audio_files = [
    filename
    for filename in uploaded
    if Path(filename).suffix.lower() in SUPPORTED_EXTENSIONS
]

if not audio_files:
    raise ValueError("No supported audio files were uploaded.")

print(f"Uploaded {len(audio_files)} audio file(s):")
for filename in audio_files:
    print("-", filename)


Saving query05_D.wav to query05_D.wav
Saving query04_D.wav to query04_D.wav
Saving query03_D.wav to query03_D.wav
Saving query02_D.wav to query02_D.wav
Saving query01_D.wav to query01_D.wav
Saving query05_C.wav to query05_C.wav
Saving query04_C.wav to query04_C.wav
Saving query03_C.wav to query03_C.wav
Saving query02_C.wav to query02_C.wav
Saving query01_C.wav to query01_C.wav
Saving query05_B.wav to query05_B.wav
Saving query04_B.wav to query04_B.wav
Saving query03_B.wav to query03_B.wav
Saving query02_B.wav to query02_B.wav
Saving query01_B.wav to query01_B.wav
Saving query05_A.wav to query05_A.wav
Saving query04_A.wav to query04_A.wav
Saving query03_A.wav to query03_A.wav
Saving query02_A.wav to query02_A.wav
Saving query01_A.wav to query01_A.wav
Saving query05_original.wav to query05_original.wav
Saving query04_original.wav to query04_original.wav
Saving query03_original.wav to query03_original.wav
Saving query02_original.wav to query02_original.wav
Saving query01_original.wav to q

## 4. Define the pipeline-facing transcription function


In [4]:
from pathlib import Path
from typing import Any


def transcribe(audio_file: str | Path) -> dict[str, Any]:
    """Transcribe one short English product-query audio file."""
    audio_path = Path(audio_file)

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    segments, info = model.transcribe(
        str(audio_path),
        language="en",
        beam_size=5,
        vad_filter=True,
        word_timestamps=True,
        condition_on_previous_text=False,
    )

    transcript_parts: list[str] = []
    segment_results: list[dict[str, Any]] = []

    # Iterating over segments performs the transcription.
    for segment in segments:
        segment_text = segment.text.strip()

        if segment_text:
            transcript_parts.append(segment_text)

        words = [
            {
                "word": word.word.strip(),
                "start": word.start,
                "end": word.end,
                "confidence": word.probability,
            }
            for word in (segment.words or [])
        ]

        segment_results.append(
            {
                "start": segment.start,
                "end": segment.end,
                "text": segment_text,
                "words": words,
            }
        )

    return {
        "text": " ".join(transcript_parts).strip(),
        "language": info.language,
        "language_probability": info.language_probability,
        "segments": segment_results,
        "source_file": str(audio_path),
    }


## 5. Transcribe all uploaded files


In [5]:
transcription_results = []

for audio_file in audio_files:
    output = transcribe(audio_file)
    transcription_results.append(output)

    print(f"\n{audio_file}")
    print("-" * len(audio_file))
    print(output["text"])
    print(
        f"Language: {output['language']} "
        f"({output['language_probability']:.1%} confidence)"
    )



query05_D.wav
-------------
Find a 32-ounce hydro flask under $40.
Language: en (100.0% confidence)

query04_D.wav
-------------
Want a CRB moisturizer with SPF 30?
Language: en (100.0% confidence)

query03_D.wav
-------------
Samsung Galaxy S25 case under $25.
Language: en (100.0% confidence)

query02_D.wav
-------------
You a logitech MX Master 3s mouse under $80?
Language: en (100.0% confidence)

query01_D.wav
-------------
An eco-friendly stainless steel cleaner under $15.
Language: en (100.0% confidence)

query05_C.wav
-------------
Find a 32-ounce hydro flask under $40.
Language: en (100.0% confidence)

query04_C.wav
-------------
Wistarizer with SPF 30
Language: en (100.0% confidence)

query03_C.wav
-------------
Show me a Samsung Galaxy S25 case under $25.
Language: en (100.0% confidence)

query02_C.wav
-------------
Logitech MX Master 3S mouse under $80.
Language: en (100.0% confidence)

query01_C.wav
-------------
I need an eco-friendly stainless steel cleaner under $15.
Lan

## 6. Review transcripts and timestamps


In [6]:
import pandas as pd

transcripts_df = pd.DataFrame(
    {
        "file": result["source_file"],
        "transcript": result["text"],
        "language": result["language"],
        "language_confidence": result["language_probability"],
    }
    for result in transcription_results
)

transcripts_df


,file,transcript,language,language_confidence
0,query05_D.wav,Find a 32-ounce hydro flask under $40.,en,1
1,query04_D.wav,Want a CRB moisturizer with SPF 30?,en,1
2,query03_D.wav,Samsung Galaxy S25 case under $25.,en,1
3,query02_D.wav,You a logitech MX Master 3s mouse under $80?,en,1
4,query01_D.wav,An eco-friendly stainless steel cleaner under ...,en,1
5,query05_C.wav,Find a 32-ounce hydro flask under $40.,en,1
6,query04_C.wav,Wistarizer with SPF 30,en,1
7,query03_C.wav,Show me a Samsung Galaxy S25 case under $25.,en,1
8,query02_C.wav,Logitech MX Master 3S mouse under $80.,en,1
9,query01_C.wav,I need an eco-friendly stainless steel cleaner...,en,1


In [7]:
# Inspect word timestamps and confidence for one uploaded file.
selected_result = transcription_results[0]

print("File:", selected_result["source_file"])
print("Transcript:", selected_result["text"])

for segment in selected_result["segments"]:
    print(f"\n[{segment['start']:.2f}s–{segment['end']:.2f}s] {segment['text']}")
    for word in segment["words"]:
        print(
            f"  {word['word']:<20} "
            f"{word['start']:>6.2f}s–{word['end']:>6.2f}s "
            f"confidence={word['confidence']:.3f}"
        )


File: query05_D.wav
Transcript: Find a 32-ounce hydro flask under $40.

[0.43s–3.81s] Find a 32-ounce hydro flask under $40.
  Find                   0.43s–  1.01s confidence=0.945
  a                      1.01s–  1.21s confidence=0.983
  32                     1.21s–  1.77s confidence=0.959
  -ounce                 1.77s–  2.11s confidence=0.711
  hydro                  2.11s–  2.53s confidence=0.687
  flask                  2.53s–  3.03s confidence=0.867
  under                  3.03s–  3.37s confidence=0.980
  $40.                   3.37s–  3.81s confidence=0.977


## 7. Add reference transcripts


In [8]:
original_reference_queries = {
    "query01.wav": (
        "Recommend an eco-friendly stainless-steel cleaner "
        "under fifteen dollars."
    ),
    "query02.wav": (
        "Compare Weiman with Therapy Clean."
    ),
    "query03.wav": (
        "Show me Seventh Generation products."
    ),
    "query04.wav": (
        "Find a plant-based kitchen cleaner."
    ),
    "query05.wav": (
        "Show products under twelve ninety-nine."
    ),
    "query06.wav": (
        "Compare Method and Mrs. Meyer's."
    ),
    "query07.wav": (
        "Find fragrance-free cleaner."
    ),
    "query08.wav": (
        "Highest rated stainless steel cleaner."
    ),
    "query09.wav": (
        "Best cleaner for granite countertops."
    ),
    "query10.wav": (
        "Show products with four-point-five stars or higher."
    )
}

print("Reference transcripts added:", len(original_reference_queries))


Reference transcripts added: 10


In [9]:
voicemod_reference_queries = {
    "query01.wav":
        "I need an eco-friendly stainless-steel cleaner under fifteen dollars.",

    "query02.wav":
        "Find me a Logitech MX Master 3S mouse under eighty dollars.",

    "query03.wav":
        "Show me a Samsung Galaxy S25 case under twenty-five dollars.",

    "query04.wav":
        "I want a CeraVe moisturizer with SPF 30.",

    "query05.wav":
        "Find a 32-ounce Hydro Flask under forty dollars.",
}

print("Reference transcripts added:", len(voicemod_reference_queries))

Reference transcripts added: 5


## 8. Evaluate word error rate


In [10]:
from pathlib import Path

def parse_audio_filename(filename):
    """
    Supports:
      query01.wav
      query01_original.wav
      query01_A.wav
      query01_B.wav
      query01_C.wav
    """

    stem = Path(filename).stem
    parts = stem.split("_")

    query_name = parts[0]

    if len(parts) == 1:
        speaker = "original"
    else:
        speaker = "_".join(parts[1:])

        # Treat explicitly labeled original files as baseline too
        if speaker.lower() == "original":
            speaker = "original"

    return query_name, speaker

In [11]:
import re

def normalize_text(text: str) -> str:
    text = text.lower().strip()

    # Normalize equivalent numeric forms
    replacements = {
        "$12.99": "twelve ninety nine",
        "12.99": "twelve ninety nine",

        "$15": "fifteen dollars",
        "15 dollars": "fifteen dollars",

        "$25": "twenty five dollars",
        "25 dollars": "twenty five dollars",

        "$40": "forty dollars",
        "40 dollars": "forty dollars",

        "$80": "eighty dollars",
        "80 dollars": "eighty dollars",

        "32 oz": "32 ounce",
        "32oz": "32 ounce",
        "32-ounce": "32 ounce",
        "32 ounce": "32 ounce",

        "4.5": "four point five",
        "4.5 stars": "four point five stars",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # Normalize punctuation/hyphens
    text = text.replace("-", " ")
    text = text.replace("–", " ")
    text = text.replace("—", " ")
    text = re.sub(r"[’']", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [12]:
from jiwer import wer
from pathlib import Path
import pandas as pd

evaluation_results = []

for result in transcription_results:

    filename = Path(result["source_file"]).name
    stem = Path(filename).stem

    # Ignore duplicate Colab uploads like query01 (1).wav
    if re.search(r" \(\d+\)\.wav$", filename, re.IGNORECASE):
        print(f"Skipping duplicate upload: {filename}")
        continue

    query_name, speaker = parse_audio_filename(filename)
    query_key = f"{query_name}.wav"

    # ---------------------------------
    # Choose the correct reference set
    # ---------------------------------

    if "_" in stem:
        # Voicemod:
        # query01_original.wav
        # query01_A.wav, etc.
        reference_set = "voicemod"
        references = voicemod_reference_queries

    else:
        # Original prototype:
        # query01.wav
        reference_set = "prototype"
        references = original_reference_queries

    if query_key not in references:
        print(
            f"Skipping {filename}: "
            f"no {reference_set} reference for {query_key}"
        )
        continue

    reference = references[query_key]
    transcript = result["text"]

    evaluation_results.append({
        "file": filename,
        "dataset": reference_set,
        "speaker": speaker,
        "query": query_name,
        "reference": reference,
        "transcript": transcript,
        "wer": wer(
            normalize_text(reference),
            normalize_text(transcript)
        )
    })

evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df)

,file,dataset,speaker,query,reference,transcript,wer
0,query05_D.wav,voicemod,D,query05,Find a 32-ounce Hydro Flask under forty dollars.,Find a 32-ounce hydro flask under $40.,0.000000
1,query04_D.wav,voicemod,D,query04,I want a CeraVe moisturizer with SPF 30.,Want a CRB moisturizer with SPF 30?,0.250000
2,query03_D.wav,voicemod,D,query03,Show me a Samsung Galaxy S25 case under twenty...,Samsung Galaxy S25 case under $25.,0.272727
3,query02_D.wav,voicemod,D,query02,Find me a Logitech MX Master 3S mouse under ei...,You a logitech MX Master 3s mouse under $80?,0.181818
4,query01_D.wav,voicemod,D,query01,I need an eco-friendly stainless-steel cleaner...,An eco-friendly stainless steel cleaner under ...,0.181818
5,query05_C.wav,voicemod,C,query05,Find a 32-ounce Hydro Flask under forty dollars.,Find a 32-ounce hydro flask under $40.,0.000000
6,query04_C.wav,voicemod,C,query04,I want a CeraVe moisturizer with SPF 30.,Wistarizer with SPF 30,0.625000
7,query03_C.wav,voicemod,C,query03,Show me a Samsung Galaxy S25 case under twenty...,Show me a Samsung Galaxy S25 case under $25.,0.000000
8,query02_C.wav,voicemod,C,query02,Find me a Logitech MX Master 3S mouse under ei...,Logitech MX Master 3S mouse under $80.,0.272727
9,query01_C.wav,voicemod,C,query01,I need an eco-friendly stainless-steel cleaner...,I need an eco-friendly stainless steel cleaner...,0.000000


## 9. Check critical product terms


In [13]:
from pathlib import Path
import pandas as pd

def check_terms(transcript: str, expected_terms: list[str]) -> dict[str, bool]:
    """Check whether expected product terms appear in the transcript."""

    normalized_transcript = normalize_text(transcript)

    return {
        term: normalize_text(term) in normalized_transcript
        for term in expected_terms
    }


# Expected keywords for each audio file.
original_terms_by_query = {
    "query01.wav": [
        "eco-friendly",
        "stainless-steel",
        "fifteen dollars"
    ],
    "query02.wav": [
        "weiman",
        "therapy clean"
    ],
    "query03.wav": [
        "seventh generation"
    ],
    "query04.wav": [
        "plant-based",
        "kitchen cleaner"
    ],
    "query05.wav": [
        "twelve ninety-nine"
    ],
    "query06.wav": [
        "method",
        "mrs. meyer's"
    ],
    "query07.wav": [
        "fragrance-free"
    ],
    "query08.wav": [
        "stainless steel cleaner"
    ],
    "query09.wav": [
        "granite countertops"
    ],
    "query10.wav": [
        "four-point-five stars"
    ]
}

voicemod_terms_by_query = {
    "query01.wav": [
        "eco-friendly",
        "stainless-steel",
        "fifteen dollars"
    ],
    "query02.wav": [
        "logitech",
        "mx master 3s",
        "eighty dollars"
    ],
    "query03.wav": [
        "samsung galaxy s25",
        "twenty-five dollars"
    ],
    "query04.wav": [
        "cerave",
        "spf 30"
    ],
    "query05.wav": [
        "32-ounce",
        "hydro flask",
        "forty dollars"
    ],
}

def check_terms(transcript, expected_terms):
    normalized_transcript = normalize_text(transcript)

    return {
        term: normalize_text(term) in normalized_transcript
        for term in expected_terms
    }


term_records = []

for result in transcription_results:

    filename = Path(result["source_file"]).name
    stem = Path(filename).stem

    query_name, speaker = parse_audio_filename(filename)
    query_key = f"{query_name}.wav"

    # Voicemod files have a suffix:
    # query01_original.wav
    # query01_A.wav
    # query01_B.wav, etc.
    if "_" in stem:
        dataset = "voicemod"
        expected_terms = voicemod_terms_by_query.get(query_key)

    else:
        dataset = "prototype"
        expected_terms = original_terms_by_query.get(query_key)

    if not expected_terms:
        # Expected for prototype query06-query10 if
        # product terms were only defined for queries 01-05
        continue

    matches = check_terms(
        result["text"],
        expected_terms
    )

    for term, matched in matches.items():

        term_records.append({
            "file": filename,
            "dataset": dataset,
            "speaker": speaker,
            "query": query_name,
            "expected_term": term,
            "recognized": matched,
        })

term_accuracy_df = pd.DataFrame(term_records)

display(term_accuracy_df)

,file,dataset,speaker,query,expected_term,recognized
0,query05_D.wav,voicemod,D,query05,32-ounce,True
1,query05_D.wav,voicemod,D,query05,hydro flask,True
2,query05_D.wav,voicemod,D,query05,forty dollars,True
3,query04_D.wav,voicemod,D,query04,cerave,False
4,query04_D.wav,voicemod,D,query04,spf 30,True
...,...,...,...,...,...,...
75,query02.wav,prototype,original,query02,weiman,False
76,query02.wav,prototype,original,query02,therapy clean,True
77,query01.wav,prototype,original,query01,eco-friendly,True
78,query01.wav,prototype,original,query01,stainless-steel,True


## 10. Results
### Original 10 Files

In [14]:
prototype_df = evaluation_df[
    evaluation_df["dataset"] == "prototype"
].copy()

prototype_wer = prototype_df["wer"].mean()

print(f"Original 10 files - Average WER: {prototype_wer * 100:.1f}%")

Original 10 files - Average WER: 6.5%


In [15]:
prototype_wer_table = prototype_df[
    ["file", "query", "wer"]
].copy()

prototype_wer_table["WER (%)"] = (
    prototype_wer_table["wer"] * 100
).round(1)

display(
    prototype_wer_table[
        ["file", "WER (%)"]
    ]
)

,file,WER (%)
25,query10.wav,0.0
26,query09.wav,0.0
27,query08.wav,0.0
28,query07.wav,25.0
29,query06.wav,0.0
30,query05.wav,0.0
31,query04.wav,0.0
32,query03.wav,20.0
33,query02.wav,20.0
34,query01.wav,0.0


In [16]:
prototype_term_df = term_accuracy_df[
    term_accuracy_df["dataset"] == "prototype"
].copy()

prototype_term_accuracy = (
    prototype_term_df["recognized"].mean()
)

print(
    f"Original 10 files - Product-term accuracy: "
    f"{prototype_term_accuracy * 100:.1f}%"
)

Original 10 files - Product-term accuracy: 80.0%


In [17]:
prototype_term_by_query = (
    prototype_term_df
    .groupby("query", as_index=False)["recognized"]
    .mean()
)

prototype_term_by_query["Product-term accuracy (%)"] = (
    prototype_term_by_query["recognized"] * 100
).round(1)

display(
    prototype_term_by_query[
        ["query", "Product-term accuracy (%)"]
    ]
)

,query,Product-term accuracy (%)
0,query01,100.0
1,query02,50.0
2,query03,0.0
3,query04,100.0
4,query05,100.0
5,query06,100.0
6,query07,0.0
7,query08,100.0
8,query09,100.0
9,query10,100.0


In [18]:
prototype_summary = pd.DataFrame({
    "Metric": [
        "Average WER",
        "Product-term accuracy"
    ],
    "Result (%)": [
        round(prototype_wer * 100, 1),
        round(prototype_term_accuracy * 100, 1)
    ]
})

display(prototype_summary)

,Metric,Result (%)
0,Average WER,6.5
1,Product-term accuracy,80.0


### VoiceMod

In [19]:
voicemod_errors = term_accuracy_df[
    (term_accuracy_df["dataset"] == "voicemod") &
    (term_accuracy_df["recognized"] == False)
].copy()

display(
    voicemod_errors[
        [
            "file",
            "speaker",
            "query",
            "expected_term"
        ]
    ]
)

,file,speaker,query,expected_term
3,query04_D.wav,D,query04,cerave
16,query04_C.wav,C,query04,cerave
26,query05_B.wav,B,query05,32-ounce
29,query04_B.wav,B,query04,cerave
31,query03_B.wav,B,query03,samsung galaxy s25
39,query05_A.wav,A,query05,32-ounce
40,query05_A.wav,A,query05,hydro flask
42,query04_A.wav,A,query04,cerave
46,query02_A.wav,A,query02,logitech
47,query02_A.wav,A,query02,mx master 3s


In [20]:
term_failure_summary = (
    voicemod_errors
    .groupby("expected_term")
    .size()
    .reset_index(name="miss_count")
    .sort_values("miss_count", ascending=False)
)

display(term_failure_summary)

,expected_term,miss_count
1,cerave,5
0,32-ounce,2
2,hydro flask,1
3,logitech,1
4,mx master 3s,1
5,samsung galaxy s25,1


In [21]:
speaker_failure_summary = (
    voicemod_errors
    .groupby("speaker")
    .size()
    .reset_index(name="miss_count")
    .sort_values("miss_count", ascending=False)
)

display(speaker_failure_summary)

,speaker,miss_count
0,A,5
1,B,3
2,C,1
3,D,1
4,original,1


In [22]:
voicemod_transcription_errors = evaluation_df[
    evaluation_df["dataset"] == "voicemod"
].copy()

display(
    voicemod_transcription_errors[
        [
            "file",
            "speaker",
            "reference",
            "transcript",
            "wer"
        ]
    ].sort_values("wer", ascending=False)
)

,file,speaker,reference,transcript,wer
15,query05_A.wav,A,Find a 32-ounce Hydro Flask under forty dollars.,You'll bounce Hydroflask under $40.,0.666667
6,query04_C.wav,C,I want a CeraVe moisturizer with SPF 30.,Wistarizer with SPF 30,0.625000
12,query03_B.wav,B,Show me a Samsung Galaxy S25 case under twenty...,Sunkaku CS25 case under $25.,0.545455
11,query04_B.wav,B,I want a CeraVe moisturizer with SPF 30.,1 A0V Moisturizer with SPF 30,0.500000
16,query04_A.wav,A,I want a CeraVe moisturizer with SPF 30.,Is there a V-moisturizer with SPF 30?,0.375000
21,query04_original.wav,original,I want a CeraVe moisturizer with SPF 30.,I want to serve e-moisturizer with SPF 30.,0.375000
10,query05_B.wav,B,Find a 32-ounce Hydro Flask under forty dollars.,32 hours hydro flask under $40,0.333333
8,query02_C.wav,C,Find me a Logitech MX Master 3S mouse under ei...,Logitech MX Master 3S mouse under $80.,0.272727
2,query03_D.wav,D,Show me a Samsung Galaxy S25 case under twenty...,Samsung Galaxy S25 case under $25.,0.272727
18,query02_A.wav,A,Find me a Logitech MX Master 3S mouse under ei...,Find me in Clojatec MS Master 3S mouse under $80.,0.272727


In [23]:
voicemod_term_df = term_accuracy_df[
    term_accuracy_df["dataset"] == "voicemod"
].copy()

product_accuracy_by_speaker = (
    voicemod_term_df
    .groupby("speaker", as_index=False)["recognized"]
    .mean()
)

product_accuracy_by_speaker[
    "Product-term accuracy (%)"
] = (
    product_accuracy_by_speaker["recognized"] * 100
).round(1)

display(
    product_accuracy_by_speaker[
        ["speaker", "Product-term accuracy (%)"]
    ]
)

,speaker,Product-term accuracy (%)
0,A,61.5
1,B,76.9
2,C,92.3
3,D,92.3
4,original,92.3


In [24]:
voicemod_df = evaluation_df[
    evaluation_df["dataset"] == "voicemod"
].copy()

wer_by_speaker = (
    voicemod_df
    .groupby("speaker", as_index=False)["wer"]
    .mean()
)

wer_by_speaker["WER (%)"] = (
    wer_by_speaker["wer"] * 100
).round(1)

display(
    wer_by_speaker[
        ["speaker", "WER (%)"]
    ]
)

,speaker,WER (%)
0,A,28.1
1,B,33.0
2,C,18.0
3,D,17.7
4,original,7.5
